# Hematology Domain-Expert LLM — QLoRA fine-tune (Llama-3.1-8B)

Companion notebook for **A Hybrid Multimodal Lab Assistant** (BSc thesis).

Runs on a free Colab **T4 (16 GB)** in ~2.5–3 h for 3 epochs over ~2 500 samples.

Inputs you must provide before running:
1. `train.jsonl` produced locally by `scripts/build_finetune_dataset.py` (uploaded to this Colab session OR pushed to a private HF dataset).
2. A HuggingFace token (write access) — `hf_…` — to push the LoRA adapter back.

Outputs:
* LoRA adapter on HuggingFace Hub (~50 MB) you can serve with **vLLM** locally.
* `OPENAI_BASE_URL` + `llm.model_name` settings to plug into your repo's `config.yaml`.

## 1. Environment

In [ ]:
# Confirm GPU
!nvidia-smi

In [ ]:
%pip install -q -U pip
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Pin a TRL/transformers combo that uses the new SFTConfig + processing_class API
# AND satisfies Unsloth's minimum (transformers >= 4.50.3 for Qwen3 support).
%pip install -q -U "trl>=0.12,<0.15" "transformers>=4.50.3,<4.55" peft accelerate bitsandbytes datasets
print('Installed. RESTART THE RUNTIME NOW: Runtime -> Restart session, then re-run from cell 3.')

## 2. Load the base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print('Base model ready:', BASE_MODEL)

## 3. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
    bias='none',
)

## 4. Load the dataset

Pick **one** of the two cells below.

In [ ]:
# OPTION A — upload train.jsonl from your laptop directly to Colab
from google.colab import files  # type: ignore
uploaded = files.upload()  # choose train.jsonl
import os, json
FNAME = next(iter(uploaded))
print('uploaded:', FNAME, os.path.getsize(FNAME), 'bytes')
from datasets import load_dataset
raw_ds = load_dataset('json', data_files=FNAME, split='train')
print(raw_ds)

In [ ]:
# OPTION B — pull from a (private) HuggingFace dataset repo
# from huggingface_hub import login; login('hf_xxx')
# from datasets import load_dataset
# raw_ds = load_dataset('YOU/hematology-sft', split='train')
# print(raw_ds)

In [ ]:
# Render with chat template AND tokenize up-front. This avoids both
# (a) "too many dimensions 'str'" from a string column reaching the collator, and
# (b) Unsloth's wrapper insisting on a `text` field.
def render(example):
    text = tokenizer.apply_chat_template(example['messages'], tokenize=False)
    enc = tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH)
    enc['text'] = text  # keep for Unsloth's sanity check
    return enc

ds = raw_ds.map(render, remove_columns=raw_ds.column_names)
ds = ds.shuffle(seed=42)
print(ds)
print('---SAMPLE text (first 600 chars)---')
print(ds[0]['text'][:600])
print('input_ids length:', len(ds[0]['input_ids']))

## 5. Train

In [ ]:
# Uses the new TRL SFTConfig + processing_class API (TRL >= 0.12).
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
import trl
print('TRL version:', trl.__version__)

cfg = SFTConfig(
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    dataset_kwargs={'skip_prepare_dataset': True},
    remove_unused_columns=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    output_dir='outputs',
    save_strategy='epoch',
    report_to='none',
)

# Strip the `text` string column NOW that SFTConfig has captured its name.
# The collator must only see `input_ids` / `attention_mask`.
train_ds = ds.remove_columns([c for c in ds.column_names if c not in ('input_ids', 'attention_mask')])
print('Final columns going into trainer:', train_ds.column_names)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    data_collator=collator,
    args=cfg,
)

trainer_stats = trainer.train()
trainer_stats

## 6. Quick sanity inference

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {'role': 'system', 'content': 'You are a clinical hematology expert. Cite sources.'},
    {'role': 'user', 'content': 'A peripheral smear shows 78% neutrophils with toxic granulation and CBC reports WBC 18.0 x10^9/L. Interpret.'},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.2, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 7. Save & push to HuggingFace Hub

In [ ]:
# === Save the LoRA adapter locally (always runs) ===
model.save_pretrained('hema-llama-lora')
tokenizer.save_pretrained('hema-llama-lora')
print('Saved local adapter to ./hema-llama-lora (~50 MB)')

In [ ]:
# === Push to HuggingFace Hub ===
# Run this cell, paste your hf_... write token when prompted, pick a username + repo name.
# Get a token at: https://huggingface.co/settings/tokens  (role: WRITE)
import getpass
from huggingface_hub import login, whoami, create_repo, HfApi

HF_USERNAME = input('Your HuggingFace username (e.g. johndoe): ').strip()
REPO_NAME   = input('Repo name [hematology-llama-3.1-8b-lora]: ').strip() or 'hematology-llama-3.1-8b-lora'
PRIVATE     = (input('Private repo? [Y/n]: ').strip().lower() or 'y') == 'y'
HF_TOKEN    = getpass.getpass('Paste your HF write token (hf_...): ').strip()

login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in as:', whoami()['name'])

repo_id = f'{HF_USERNAME}/{REPO_NAME}'
create_repo(repo_id, private=PRIVATE, exist_ok=True, token=HF_TOKEN)
print('Repo ready:', repo_id)

# Upload the adapter + tokenizer
model.push_to_hub(repo_id, token=HF_TOKEN, private=PRIVATE)
tokenizer.push_to_hub(repo_id, token=HF_TOKEN, private=PRIVATE)
print(f'\n✅ Done. View at: https://huggingface.co/{repo_id}')
print(f'\nUse this in config.yaml -> llm.model_name:\n  {repo_id}')

## 8. Plug back into the repo (no code changes)

On a machine with a GPU (your laptop / RunPod / Modal):

```bash
pip install vllm
vllm serve YOU/hematology-llama-3.1-8b-lora --port 8001 --enable-lora
```

Then in `config.yaml`:
```yaml
llm:
  model_name: YOU/hematology-llama-3.1-8b-lora
  temperature: 0.1
```
And in `.env`:
```
OPENAI_API_KEY=dummy
OPENAI_BASE_URL=http://localhost:8001/v1
```

Stage 3 of the pipeline now talks to your domain-expert model instead of GPT-4o.